# Robotwin VV attention plotting pipeline

The plotting implementation lives in `render_vv_attention.py`; this notebook renders the raw joint-softmax VV slice from `attn-exp-vv-10-6/step_024`. The display applies 6×6 block-max pooling followed by `PowerNorm(gamma=0.25)`: zero remains pure white, small positive probabilities become visible, and unavailable future positions remain light gray. Within each layer, all 24 heads share one exact raw-probability colorbar from zero to that layer's finite maximum, with no percentile clipping.

Matrix orientation:

- rows: concatenated video Q tokens, small to large from top to bottom;
- columns: visible video K tokens, including past and current chunks, small to large from left to right;
- gray cells: future K fields that were empty in the CSV;
- the preview exports one ultra-high-resolution 4×6 head grid;
- figures are written under `<experiment>/<step_XXX>/layerN/`.

The helper uses only NumPy and Matplotlib; it does not import PyTorch.

In [ ]:
from pathlib import Path
import sys


def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if all((candidate / name).is_dir() for name in ("notebooks", "data", "figures")):
            return candidate
    raise FileNotFoundError("Could not find jupyter-plot root")


PROJECT_ROOT = find_project_root()
WORKSET_NAME = "26Aug9-Robotwin-VV-attention"
HELPER_DIR = PROJECT_ROOT / "notebooks" / WORKSET_NAME
sys.path.insert(0, str(HELPER_DIR))

from render_vv_attention import (
    attention_step_summary,
    discover_attention_steps,
    layer_probability_vmax,
    load_head_matrix,
    locate_experiment,
    metadata,
    output_dir,
    render_attention_steps,
    render_layer_grid,
    render_head,
)

print(f"project : {PROJECT_ROOT}")
print(f"workset : {WORKSET_NAME}")

## Parameters

In [ ]:
ATTENTION_LINK = "attn-exp-vv-10-6"
EXPERIMENT_SLUG = None
ATTENTION_STEPS = ("step_024",)
LAYERS_TO_RENDER = (15,)
FORMATS = ("png",)
V_MAX = None  # None = exact maximum shared by all 24 heads within each layer.
BLOCK_MAX_SIZE = 6
POWER_GAMMA = 0.25
DPI = 1200

# Layer 15 balances high VV mass with a non-extreme per-cell dynamic range.
PREVIEW_STEP = "step_024"
PREVIEW_LAYER = 15
PREVIEW_HEAD = 0

## Inspect the linked attention output

In [ ]:
EXPERIMENT_DIR, SUMMARY = locate_experiment(PROJECT_ROOT, ATTENTION_LINK, EXPERIMENT_SLUG)
AVAILABLE_STEPS = discover_attention_steps(EXPERIMENT_DIR)
if PREVIEW_STEP not in AVAILABLE_STEPS:
    raise ValueError(f"PREVIEW_STEP {PREVIEW_STEP!r} not in {AVAILABLE_STEPS}")
PREVIEW_SUMMARY = attention_step_summary(SUMMARY, PREVIEW_STEP)
EXPECTED_SHAPE, LAYERS, NUM_HEADS, ROW_BOUNDS, HISTORY_BOUNDS = metadata(
    EXPERIMENT_DIR, SUMMARY, PREVIEW_STEP
)
PREVIEW_MATRIX_DIR = EXPERIMENT_DIR / PREVIEW_STEP
FIGURES_ROOT = PROJECT_ROOT / "figures" / WORKSET_NAME

print(f"experiment : {EXPERIMENT_DIR.name}")
print(f"steps      : {AVAILABLE_STEPS}")
print(f"shape      : {EXPECTED_SHAPE}")
print(f"chunks     : {PREVIEW_SUMMARY['num_chunks']}")
print(f"timestep   : {PREVIEW_SUMMARY['scheduler_timestep']}")
print(f"layers     : {len(LAYERS)}")
print(f"heads      : {NUM_HEADS}")
print(f"Q bounds   : {ROW_BOUNDS}")
print(f"K bounds   : {HISTORY_BOUNDS}")

## Draw one figure

This cell loads all 24 heads in `PREVIEW_LAYER` to establish the shared layer colorbar, then draws one selected head. Change `PREVIEW_LAYER` and `PREVIEW_HEAD` to inspect another head.

In [ ]:
PREVIEW_HEADS = list(range(NUM_HEADS))
LAYER_MATRICES = {
    head: load_head_matrix(PREVIEW_MATRIX_DIR, EXPECTED_SHAPE, PREVIEW_LAYER, head)
    for head in PREVIEW_HEADS
}
PREVIEW_VMAX = layer_probability_vmax(LAYER_MATRICES, V_MAX, POWER_GAMMA)
MATRIX = LAYER_MATRICES[PREVIEW_HEAD]
SMALL_DIR = output_dir(FIGURES_ROOT, EXPERIMENT_DIR, PREVIEW_LAYER, PREVIEW_STEP)
SMALL_PATHS = render_head(
    MATRIX,
    PREVIEW_LAYER,
    PREVIEW_HEAD,
    PREVIEW_VMAX,
    ROW_BOUNDS,
    HISTORY_BOUNDS,
    SMALL_DIR,
    formats=FORMATS,
    dpi=DPI,
    block_size=BLOCK_MAX_SIZE,
    power_gamma=POWER_GAMMA,
)
print(*SMALL_PATHS, sep="\n")

## Draw one layer in the reference style

This cell renders one complete ultra-high-resolution head grid for `PREVIEW_LAYER`. The 24 heads reuse `PREVIEW_VMAX`, 6×6 block-max pooling, and `PowerNorm(gamma=0.25)`.

In [ ]:
LAYER_DIR = output_dir(FIGURES_ROOT, EXPERIMENT_DIR, PREVIEW_LAYER, PREVIEW_STEP)
LAYER_PATHS = render_layer_grid(
    LAYER_MATRICES,
    PREVIEW_LAYER,
    PREVIEW_HEADS,
    PREVIEW_VMAX,
    ROW_BOUNDS,
    HISTORY_BOUNDS,
    LAYER_DIR,
    formats=FORMATS,
    columns_per_row=6,
    dpi=DPI,
    block_size=BLOCK_MAX_SIZE,
    power_gamma=POWER_GAMMA,
)
print(*LAYER_PATHS, sep="\n")

## Draw the selected step_024 layer

This render cell visits Layer 15 only and exports the 4×6 grid without generating 24 redundant single-head files.

In [ ]:
MANIFESTS = render_attention_steps(
    PROJECT_ROOT,
    attention_link=ATTENTION_LINK,
    experiment_slug=EXPERIMENT_SLUG,
    attention_steps=ATTENTION_STEPS,
    layers=LAYERS_TO_RENDER,
    layer_grid=True,
    grid_only=True,
    formats=FORMATS,
    color_vmax=V_MAX,
    dpi=DPI,
    block_size=BLOCK_MAX_SIZE,
    power_gamma=POWER_GAMMA,
)
MANIFESTS